# KONA-style Energy-Based Reasoning
## Research: Can EBM reranking improve open LLM reasoning?

**Pipeline:** Prompt → N candidates → Energy scoring → Best answer
**Strategy:** Math + Logic first (GSM8K, ARC, BBH)

In [ ]:
# Install dependencies (run once)
!pip install -q torch transformers accelerate datasets evaluate sentencepiece bitsandbytes

# Clone project repo
import os
if not os.path.exists('kona-ebm'):
    !git clone <your-repo-url> kona-ebm
%cd kona-ebm

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
print("Model loaded")

In [ ]:
# Phase 1: Baseline benchmark (single sample test)
from benchmarks.run_baseline import load_gsm8k, format_gsm8k_prompt, generate

ds = load_gsm8k("test", max_samples=5)
for i, ex in enumerate(ds):
    prompt = format_gsm8k_prompt(ex["question"])
    answer, latency = generate(model, tokenizer, prompt)
    print(f"[{i}] Latency: {latency:.2f}s")
    print(f"    Q: {ex['question'][:80]}...")
    print(f"    A: {answer[:100]}...\n")

In [ ]:
# Phase 2: KONA verifier test
from reranker.kona_verifier import KONAVerifier

verifier = KONAVerifier(MODEL_NAME)

# Test with a simple math problem
prompt = "Solve: A farmer has 12 apples. He gives 3 to his neighbor and eats 2. How many are left?"
best, score, candidates = verifier.rerank(prompt, n=3)

print(f"Prompt: {prompt}\n")
print(f"Best answer (energy={score:.4f}): {best}\n")
for i, (ans, s) in enumerate(candidates):
    print(f"  [{i}] energy={s:.4f}: {ans}")

In [ ]:
# Run full baseline benchmark (200 samples)
# !python benchmarks/run_baseline.py --model Qwen/Qwen2.5-3B-Instruct --benchmarks gsm8k --max-samples 200